# 01 Basic model: plane wave scattered by a bone sphere

| | |
|---|---|
| **Purpose** | The smallest complete OptimUS simulation: source, geometry, materials, BEM solve, field on a plane. Start here. |
| **Inputs** | 1 kHz plane wave travelling along (1, 1, 0) with unit amplitude; sphere of radius 1 m at the origin, element size 0.4 m; water outside, cortical bone inside. |
| **Outputs** | Real part and magnitude of the total pressure on the plane z = 0 (`outputs/01_basic_model_*.png`); GMRES iteration count; error against the analytical series solution. |
| **Checks** | GMRES converged; incident wave has unit amplitude everywhere outside the sphere; BEM total field within 5% (Frobenius norm) of the analytical solution. |
| **Runtime** | About 30 s full, 15 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). CI mode only uses a coarser plotting grid; the mesh is already small. |
| **Adapted from** | `notebooks/Basic model.ipynb` (upstream OptimUS). |

## Workflow

Every OptimUS simulation follows the same five steps, and the other notebooks in this folder only change one or two of them:

1. **Source**: the incident field (plane wave, piston, bowl, array).
2. **Geometry**: a closed surface mesh for each scatterer.
3. **Materials**: density, speed of sound and attenuation of each domain.
4. **Model and solve**: build the boundary integral model and solve it with GMRES.
5. **Postprocess**: evaluate the field at points (a plane or a point cloud) and plot it.

## Physics notes

Outside the sphere the total pressure is the incident plane wave plus a scattered wave,
$p = p_{inc} + p_{sc}$, and both domains satisfy the Helmholtz equation
$\nabla^2 p + k^2 p = 0$ with $k = 2\pi f / c$ in each material.
On the sphere surface the pressure and the normal particle velocity
$\frac{1}{\rho}\partial_n p$ are continuous. OptimUS turns this into boundary integral equations
on the surface only (the PMCHWT formulation) and solves the discretised system with GMRES.

For a sphere the same problem has an exact series solution (see `07_validation.ipynb`), so this
notebook can check its own answer.

> **To expand (Yash):** derive why the BEM only needs a surface mesh (Green's representation formula), and why the mesh must resolve the wavelength (rule of thumb: about 6 elements per wavelength).

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "01_basic_model"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 1000.0            # Hz
direction = (1, 1, 0)         # propagation direction, normalised by OptimUS
sphere_radius = 1.0           # m
element_size = 0.4            # m, same in CI mode (the model is already small)
grid_resolution = (100, 100) if CI_MODE else (250, 250)
bounding_box = [-3, 5, -3, 4] # x_min, x_max, y_min, y_max of the plane z = 0

## 2. Source, geometry and materials

In [ ]:
source = optimus.source.create_planewave(frequency, direction=direction)
geometry = optimus.geometry.shapes.Sphere(radius=sphere_radius, element_size=element_size)
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

wavelength_ext = material_ext.compute_wavelength(frequency)
wavelength_int = material_int.compute_wavelength(frequency)
elements_per_wavelength = wavelength_ext / element_size
print("Wavelength in water: %.3f m, in bone: %.3f m" % (wavelength_ext, wavelength_int))
print("Mesh: %d vertices, %.1f elements per wavelength in water"
      % (geometry.number_of_vertices(), elements_per_wavelength))
RESULTS["number_of_vertices"] = int(geometry.number_of_vertices())
RESULTS["elements_per_wavelength"] = round(float(elements_per_wavelength), 2)

## 3. Model and solve

In [ ]:
model = optimus.model.create_default_model(source, geometry, material_ext, material_int)
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 2)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations, "| solve time: %.2f s" % RESULTS["solve_time_s"])

## 4. Field on the plane z = 0

In [ ]:
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=grid_resolution, bounding_box=bounding_box)
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane)
save_figures(figs, NOTEBOOK_NAME)

## 5. Checks

The analytical reference is the series solution for a penetrable sphere
(`optimus.model.acoustics.Analytical`, 100 terms), evaluated on the same plane.
The 5% limit is a tolerance on a measured value: on 9 Nov 2026 the error at this mesh was 2.4%
(100 x 100 grid, CI mode) and 2.7% (250 x 250 grid). For comparison, `07_validation` gives
about 10% at element size 1.0 and under 1% at 0.1.

Note that element size 0.4 m is only 3.7 elements per wavelength in water, below the usual guideline
of about 6, and the mesh generator gives the same 118-vertex mesh for 0.4 and 0.5. The error is still
small here because the sphere is only about 1.3 wavelengths across.

In [ ]:
check("gmres_converged", converged,
      "%d iterations (max %d)" % (iterations, optimus.global_parameters.linalg.maxiter))

incident = plane.incident_field_imshow
exterior = np.isfinite(incident)
amplitude_error = np.max(np.abs(np.abs(incident[exterior]) - 1.0))
check("incident_unit_amplitude", amplitude_error < 1e-6,
      "max | |p_inc| - 1 | = %.2e outside the sphere" % amplitude_error)

total = plane.total_field_imshow
check("field_is_finite", np.all(np.isfinite(total)),
      "%d NaN or inf values on the plane" % np.sum(~np.isfinite(total)))

analytical = optimus.model.acoustics.Analytical(source, geometry, material_ext, material_int)
analytical.solve(n_iter=100)
plane_an = optimus.postprocess.VisualisePlane(analytical)
plane_an.create_computational_grid(resolution=grid_resolution, bounding_box=bounding_box)
plane_an.compute_fields()
reference = np.real(plane_an.total_field_imshow)
difference = np.nan_to_num(np.real(total) - reference)
relative_error = np.linalg.norm(difference) / np.linalg.norm(np.nan_to_num(reference))
RESULTS["relative_error_vs_analytical"] = round(float(relative_error), 4)
check("matches_analytical", relative_error < 0.05,
      "relative error (Frobenius) = %.4f, limit 0.05" % relative_error)

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))